# 책장 넘김 소리를 기준으로 MP3 일괄 분할

대상 폴더의 모든 MP3에서 기준 샘플과 유사한 책장 넘김 소리를 찾아, 각 원본 파일별 폴더에 순서대로 저장합니다.

결과 예: `output/REC1_01593/REC1_01593_001.mp3`, `002.mp3` ...

기본값은 책장 넘김 소리 자체를 결과에서 제거합니다.

In [2]:
!pip install librosa soundfile scipy numpy tqdm pydub


  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 32.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
   ------------ --------------------------- 12.6/41.9 MB 56.5 MB/s eta 0:00:01
   ----------------------- ---------------- 24.9/41.9 MB 58.5 MB/s eta 0:00:01
   ------------------------------------ --- 38.0/41.9 MB 59.0 MB/s eta 0:00:01
   ---------------------------------------- 41.9/41.9 MB 56.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ---------------------------------------- 8.3/8.3 MB 56.7 MB/s eta 0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -- -------------------------------------  1/14 [tqdm]
   -- -------------------------------------  1/14 [tqdm]
   ----------- ----------------------------  4/14 [narwhals]
   ----------- ----

In [ ]:
from pathlib import Path
import numpy as np
import librosa
from scipy.signal import find_peaks
from tqdm.auto import tqdm
import subprocess

# ===================== 사용자 설정 =====================
# 이 폴더 안에 대상 MP3와 기준 샘플을 넣으세요.
WORK_DIR = Path(r"D:\\PythonProject\\English_MP3_devide\Disney Fun to Read 1")
# CURRENT_DIR = Path(r"D:\\PythonProject\\English_MP3_devide")
INPUT_DIR = WORK_DIR / "mp3"
SAMPLE_FILE = WORK_DIR / "book_page_turn_34s.mp3"
OUTPUT_DIR = WORK_DIR / "output"

# 책장 넘김 감도: 높을수록 엄격합니다.
# SIMILARITY_THRESHOLD = 0.70
SIMILARITY_THRESHOLD = 0.90

# 같은 책장 넘김을 중복 검출하지 않을 최소 간격
MIN_PAGE_TURN_GAP = 1.0

# 검출된 책장 넘김 소리 주변에서 제거할 시간
REMOVE_BEFORE = 0.20
REMOVE_AFTER = 0.35

# 너무 짧은 결과는 저장하지 않음
MIN_OUTPUT_LENGTH = 1.0

SR = 22050
print('설정 완료')
print('대상:', INPUT_DIR)
print('기준:', SAMPLE_FILE)
print('결과:', OUTPUT_DIR)


설정 완료
대상: D:\PythonProject\English_MP3_devide\Disney Fun to Read 1\mp3
기준: D:\PythonProject\English_MP3_devide\Disney Fun to Read 1\book_page_turn_34s.mp3
결과: D:\PythonProject\English_MP3_devide\Disney Fun to Read 1\output


In [12]:
def load_audio(path):
    y, _ = librosa.load(str(path), sr=SR, mono=True)
    return y

def feature(y):
    y = y.astype(np.float32)
    if len(y) == 0:
        return np.zeros((40, 1), dtype=np.float32)
    y = y - np.mean(y)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=20, n_fft=1024, hop_length=256)
    delta = librosa.feature.delta(mfcc)
    return np.vstack([mfcc, delta])

def cosine(a, b):
    a = a.ravel(); b = b.ravel()
    n = min(len(a), len(b))
    a = a[:n]; b = b[:n]
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return 0.0 if d == 0 else float(np.dot(a, b) / d)

sample = load_audio(SAMPLE_FILE)
sample_feat = feature(sample)
sample_len = len(sample)
print(f'기준 샘플 길이: {len(sample)/SR:.2f}초')


기준 샘플 길이: 1.10초


In [13]:
def detect_page_turns(y):
    win = sample_len
    hop = int(SR * 0.10)  # 0.1초 단위로 탐색
    scores = []
    times = []

    for start in range(0, max(1, len(y) - win + 1), hop):
        chunk = y[start:start + win]
        if len(chunk) < win * 0.85:
            continue
        f = feature(chunk)
        # 평균 특징 + 전체 패턴을 함께 사용
        mean_sim = cosine(np.mean(sample_feat, axis=1), np.mean(f, axis=1))
        shape_sim = cosine(sample_feat, f)
        score = 0.65 * mean_sim + 0.35 * shape_sim
        scores.append(score)
        times.append((start + len(chunk)/2) / SR)

    if not scores:
        return []

    scores = np.asarray(scores)
    times = np.asarray(times)
    distance = max(1, int(MIN_PAGE_TURN_GAP / 0.10))
    peaks, props = find_peaks(scores, height=SIMILARITY_THRESHOLD, distance=distance)
    return [(float(times[p]), float(scores[p])) for p in peaks]


In [14]:
def export_mp3(y, start_sec, end_sec, out_file):
    start = max(0, int(start_sec * SR))
    end = min(len(y), int(end_sec * SR))
    if end <= start:
        return False
    # ffmpeg가 WAV 파이프를 받아 MP3로 저장합니다.
    import soundfile as sf
    import io
    buf = io.BytesIO()
    sf.write(buf, y[start:end], SR, format='WAV')
    result = subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-i', 'pipe:0', '-codec:a', 'libmp3lame', '-b:a', '192k', str(out_file)],
        input=buf.getvalue(), stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode(errors='ignore'))
    return True

def split_one(mp3_file):
    y = load_audio(mp3_file)
    duration = len(y) / SR
    detections = detect_page_turns(y)
    turns = sorted(t for t, _ in detections)

    # 책장 넘김 소리 자체를 제거하기 위한 구간 생성/병합
    removed = []
    for t in turns:
        a = max(0, t - REMOVE_BEFORE)
        b = min(duration, t + REMOVE_AFTER)
        if not removed or a > removed[-1][1]:
            removed.append([a, b])
        else:
            removed[-1][1] = max(removed[-1][1], b)

    segments = []
    cursor = 0.0
    for a, b in removed:
        if a - cursor >= MIN_OUTPUT_LENGTH:
            segments.append((cursor, a))
        cursor = b
    if duration - cursor >= MIN_OUTPUT_LENGTH:
        segments.append((cursor, duration))

    out_dir = OUTPUT_DIR / mp3_file.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob('*.mp3'):
        old.unlink()

    for i, (a, b) in enumerate(segments, 1):
        out_file = out_dir / f'{mp3_file.stem}_{i:03d}.mp3'
        export_mp3(y, a, b, out_file)

    return turns, len(segments)


In [15]:
# ===================== 전체 MP3 처리 =====================
if not INPUT_DIR.exists():
    raise FileNotFoundError(f'대상 폴더가 없습니다: {INPUT_DIR}')
if not SAMPLE_FILE.exists():
    raise FileNotFoundError(f'기준 샘플이 없습니다: {SAMPLE_FILE}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
files = sorted(INPUT_DIR.glob('*.mp3'))
print(f'총 {len(files)}개 MP3 처리 시작')

for f in tqdm(files, desc='전체 처리'):
    try:
        turns, count = split_one(f)
        print(f'\n{f.name}: 책장 넘김 {len(turns)}회 → 결과 {count}개')
        if turns:
            print('감지 위치:', ', '.join(f'{t:.1f}s' for t in turns))
    except Exception as e:
        print(f'\n[ERROR] {f.name}: {e}')

print('\n완료:', OUTPUT_DIR)


총 23개 MP3 처리 시작


전체 처리:   4%|▍         | 1/23 [00:05<01:54,  5.22s/it]


03 Disney Fun to Read Set 1-01 - Just Like Me Reading 2.mp3: 책장 넘김 140회 → 결과 32개
감지 위치: 1.2s, 5.0s, 6.5s, 7.5s, 8.7s, 9.7s, 10.9s, 12.2s, 14.1s, 15.9s, 17.6s, 18.9s, 20.1s, 21.2s, 22.6s, 23.9s, 24.9s, 25.9s, 26.9s, 28.6s, 29.6s, 30.6s, 31.6s, 32.8s, 33.8s, 35.0s, 36.6s, 37.8s, 38.8s, 40.8s, 42.5s, 44.0s, 45.0s, 46.4s, 47.9s, 49.0s, 50.0s, 51.5s, 52.5s, 53.6s, 54.8s, 56.1s, 57.5s, 59.0s, 60.5s, 62.0s, 63.0s, 64.5s, 65.5s, 66.8s, 68.3s, 69.3s, 70.3s, 72.3s, 73.3s, 74.8s, 76.2s, 77.8s, 79.0s, 80.5s, 82.5s, 83.5s, 84.7s, 85.8s, 87.2s, 88.7s, 89.7s, 92.0s, 93.3s, 95.3s, 96.8s, 98.2s, 99.7s, 101.2s, 102.8s, 103.8s, 104.8s, 106.2s, 107.7s, 109.2s, 110.2s, 111.2s, 112.5s, 114.5s, 116.0s, 117.5s, 118.5s, 120.2s, 122.2s, 124.2s, 125.5s, 127.0s, 128.2s, 129.9s, 131.4s, 132.6s, 133.8s, 135.1s, 136.6s, 137.6s, 138.6s, 140.2s, 141.2s, 142.3s, 143.8s, 145.2s, 146.7s, 147.8s, 148.9s, 149.9s, 151.2s, 152.2s, 153.2s, 154.7s, 156.2s, 157.3s, 158.3s, 159.4s, 160.8s, 162.3s, 163.8s, 165.6s, 166.8s, 167.9s

전체 처리:   9%|▊         | 2/23 [00:11<01:59,  5.68s/it]


03 Disney Fun to Read Set 1-02 - Bug Stew Reading 2.mp3: 책장 넘김 124회 → 결과 50개
감지 위치: 1.6s, 3.5s, 4.5s, 5.5s, 7.0s, 8.4s, 9.8s, 10.9s, 12.7s, 14.1s, 15.1s, 16.1s, 17.6s, 18.9s, 20.1s, 21.4s, 22.6s, 23.8s, 25.6s, 26.9s, 28.4s, 30.1s, 32.1s, 33.8s, 35.5s, 36.5s, 39.5s, 41.9s, 42.9s, 43.9s, 44.9s, 46.9s, 47.9s, 49.4s, 51.1s, 52.1s, 53.4s, 55.4s, 56.4s, 59.4s, 60.4s, 62.2s, 63.2s, 64.2s, 66.0s, 67.0s, 68.0s, 70.2s, 71.2s, 72.8s, 74.2s, 75.2s, 76.8s, 77.8s, 78.8s, 80.0s, 81.0s, 82.8s, 83.8s, 84.8s, 86.7s, 87.8s, 88.8s, 90.0s, 91.0s, 92.0s, 93.3s, 95.5s, 96.5s, 98.2s, 100.0s, 101.0s, 102.2s, 103.2s, 104.2s, 105.2s, 107.3s, 108.3s, 110.2s, 111.2s, 112.5s, 114.0s, 115.8s, 116.8s, 119.3s, 120.7s, 122.2s, 124.8s, 125.8s, 126.8s, 128.6s, 130.1s, 131.8s, 133.8s, 135.8s, 136.8s, 138.4s, 139.4s, 142.3s, 143.3s, 144.8s, 146.1s, 147.8s, 148.8s, 151.3s, 153.1s, 154.4s, 155.6s, 158.4s, 159.4s, 161.2s, 162.8s, 163.8s, 166.4s, 168.3s, 170.2s, 171.6s, 173.3s, 174.6s, 175.8s, 177.2s, 178.8s, 179.8s, 181.1s


...


03 Disney Fun to Read Set 1-21 - Race Around the World Reading 2.mp3: 책장 넘김 114회 → 결과 23개
감지 위치: 1.2s, 2.5s, 3.5s, 5.2s, 6.2s, 8.3s, 9.3s, 10.9s, 12.2s, 13.2s, 14.9s, 15.9s, 17.9s, 18.9s, 20.2s, 21.2s, 22.4s, 23.6s, 24.9s, 26.2s, 27.9s, 29.1s, 30.1s, 31.4s, 32.9s, 34.2s, 35.8s, 37.1s, 38.5s, 40.0s, 41.5s, 42.8s, 43.8s, 44.8s, 46.2s, 47.6s, 49.1s, 50.1s, 51.1s, 52.2s, 53.6s, 54.9s, 56.4s, 57.9s, 59.0s, 60.5s, 62.1s, 63.5s, 65.0s, 66.2s, 67.3s, 68.3s, 69.8s, 71.3s, 72.8s, 74.0s, 75.7s, 77.0s, 78.5s, 79.8s, 81.0s, 82.8s, 84.0s, 85.8s, 87.0s, 88.5s, 90.0s, 91.3s, 93.0s, 94.2s, 95.7s, 97.0s, 98.3s, 99.3s, 100.5s, 102.0s, 103.3s, 104.5s, 106.0s, 107.5s, 108.5s, 109.8s, 111.2s, 112.7s, 114.0s, 115.0s, 116.3s, 117.7s, 119.0s, 120.5s, 121.7s, 123.0s, 124.0s, 125.0s, 126.2s, 127.7s, 129.1s, 130.2s, 132.2s, 133.8s, 135.1s, 136.6s, 137.9s, 139.2s, 140.9s, 143.2s, 145.6s, 146.8s, 148.2s, 150.2s, 152.1s, 153.7s, 154.7s, 155.7s


전체 처리:  96%|█████████▌| 22/23 [02:05<00:05,  5.24s/it]


03 Disney Fun to Read Set 1-22 - Big Bear, Little Bear Reading 2.mp3: 책장 넘김 129회 → 결과 56개
감지 위치: 1.4s, 4.5s, 5.5s, 6.7s, 7.7s, 9.3s, 10.9s, 12.2s, 14.2s, 15.9s, 17.8s, 19.1s, 20.1s, 21.2s, 22.4s, 23.4s, 24.6s, 26.9s, 27.9s, 29.6s, 30.6s, 31.8s, 32.9s, 33.9s, 35.8s, 37.9s, 38.9s, 40.5s, 41.5s, 43.4s, 44.4s, 46.2s, 48.0s, 50.5s, 52.2s, 54.1s, 56.1s, 57.1s, 58.4s, 60.1s, 61.5s, 63.4s, 64.3s, 66.2s, 67.3s, 68.3s, 69.8s, 70.8s, 71.8s, 73.7s, 74.7s, 76.5s, 77.7s, 79.0s, 80.8s, 82.7s, 83.7s, 84.7s, 86.7s, 88.2s, 89.8s, 91.8s, 92.8s, 94.5s, 95.8s, 97.5s, 99.0s, 100.0s, 101.0s, 102.0s, 103.5s, 105.0s, 106.8s, 107.8s, 109.0s, 110.0s, 111.0s, 112.7s, 114.2s, 116.2s, 117.2s, 118.7s, 120.0s, 121.8s, 123.8s, 124.8s, 125.8s, 126.8s, 128.3s, 130.1s, 131.7s, 132.7s, 133.8s, 135.7s, 136.7s, 138.6s, 139.6s, 141.1s, 142.1s, 143.1s, 144.8s, 146.2s, 148.2s, 149.2s, 150.8s, 152.1s, 153.9s, 155.3s, 156.7s, 158.8s, 159.8s, 160.8s, 161.8s, 163.4s, 165.2s, 167.2s, 168.2s, 170.1s, 172.2s, 173.8s, 174.8s, 176.1s,

전체 처리: 100%|██████████| 23/23 [02:10<00:00,  5.67s/it]


03 Disney Fun to Read Set 1-23 - Fast Kart, Slow Kart Reading 2.mp3: 책장 넘김 125회 → 결과 45개
감지 위치: 1.2s, 2.5s, 3.5s, 4.7s, 5.7s, 7.5s, 8.7s, 9.8s, 10.9s, 12.2s, 14.1s, 15.9s, 17.4s, 18.6s, 20.4s, 21.4s, 22.6s, 23.9s, 25.4s, 27.8s, 28.8s, 29.9s, 31.4s, 32.5s, 34.0s, 35.0s, 36.0s, 37.1s, 38.1s, 39.1s, 41.0s, 42.8s, 44.0s, 45.0s, 46.5s, 47.6s, 48.6s, 49.6s, 51.1s, 52.2s, 54.2s, 56.1s, 57.1s, 59.0s, 60.0s, 61.5s, 62.5s, 64.7s, 65.7s, 66.7s, 67.8s, 68.8s, 69.8s, 70.8s, 72.8s, 74.3s, 75.8s, 77.8s, 78.8s, 80.5s, 81.8s, 83.0s, 84.7s, 86.2s, 87.2s, 89.2s, 90.2s, 91.8s, 93.3s, 94.5s, 96.2s, 97.3s, 98.3s, 99.3s, 100.3s, 101.3s, 103.0s, 104.2s, 106.2s, 107.8s, 108.8s, 109.8s, 110.8s, 112.8s, 114.5s, 116.0s, 117.7s, 118.8s, 120.0s, 121.0s, 122.0s, 123.0s, 124.5s, 126.2s, 127.7s, 129.1s, 130.2s, 131.9s, 132.9s, 134.3s, 135.4s, 137.1s, 138.9s, 139.9s, 141.8s, 142.9s, 144.2s, 145.9s, 147.2s, 148.2s, 149.3s, 150.3s, 151.3s, 153.1s, 154.4s, 155.4s, 157.8s, 159.7s, 161.1s, 162.8s, 164.8s, 166.7s, 168.2s, 1

## 튜닝

`SIMILARITY_THRESHOLD = 0.70`에서 시작하세요.

- 책장 넘김을 놓치면 `0.65` → `0.60`으로 낮춥니다.
- 엉뚱한 소리까지 잡으면 `0.75` → `0.80`으로 높입니다.
- 책장 소리가 결과에 남으면 `REMOVE_BEFORE`, `REMOVE_AFTER`를 조금 늘립니다.

※ `ffmpeg`가 PC에 설치되어 있어야 합니다. Windows에서는 `ffmpeg`가 PATH에 등록되어 있어야 합니다.